In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import time
import random
import pandas as pd

BASE_DOMAIN = 'https://lifehacker.ru'
TOPIC_URL = 'https://lifehacker.ru/topics/technology/'

def fetch_soup(url, session=None, retries=2, backoff=1.0):
    if session is None:
        session = requests.Session()
    for attempt in range(retries + 1):
        try:
            r = session.get(url, timeout=15, headers={
                'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                              'AppleWebKit/537.36 (KHTML, like Gecko) '
                              'Chrome/115.0.0.0 Safari/537.36'
            })
            if r.status_code == 200:
                return BeautifulSoup(r.text, 'html.parser')
            else:
                time.sleep(backoff * (2 ** attempt))
        except Exception:
            time.sleep(backoff * (2 ** attempt))
    return None

def collect_list_page_links(soup):
    links = set()
    # Несколько кандидатовых селекторов для карточек статей
    candidates = [
        'article h2 a',
        'h2 a',
        'div.post__title a',
        'a.card-title',
        'a.title',
        'a.entry-title',
    ]
    for sel in candidates:
        for a in soup.select(sel):
            href = a.get('href')
            if not href:
                continue
            if href.startswith('#'):
                continue
            full = href if href.startswith('http') else urljoin(BASE_DOMAIN, href)
            links.add(full)
    # Фолбэк: любые ссылки на lifehacker.ru внутри страницы
    if not links:
        for a in soup.find_all('a', href=True):
            href = a['href']
            full = href if href.startswith('http') else urljoin(BASE_DOMAIN, href)
            if full.startswith(BASE_DOMAIN):
                links.add(full)
    return list(links)

def extract_title_and_text(article_soup):
    # Заголовок: пытаемся несколько вариантов
    title = ''
    for sel in ['h1', 'h1.site__title', 'h1.entry-title', 'h1.post-title']:
        el = article_soup.select_one(sel)
        if el and el.get_text(strip=True):
            title = el.get_text(strip=True)
            break
    if not title and article_soup.title:
        title = article_soup.title.get_text(strip=True)

    # Текст статьи: ищем главный контейнер и собираем текст из параграфов
    content = []
    content_selectors = [
        'article', 'div.article-content', 'div.post-content', 'div.article__body',
        'div.entry-content', 'div.content', 'section.article-content'
    ]
    for sel in content_selectors:
        container = article_soup.select_one(sel)
        if container:
            for p in container.find_all(['p'], recursive=True):
                t = p.get_text(strip=True)
                if t:
                    content.append(t)
            if content:
                break

    # Ф fallback: все <p> на странице
    if not content:
        for p in article_soup.find_all('p'):
            t = p.get_text(strip=True)
            if t:
                content.append(t)

    text = '\n'.join(content)
    return title, text

def main():
    session = requests.Session()
    session.headers.update({
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                      'AppleWebKit/537.36 (KHTML, like Gecko) '
                      'Chrome/115.0.0.0 Safari/537.36'
    })

    # 1) Собираем рабочие страницы рубрики (первые 10)
    page_urls = []
    for page in range(1, 11):  # 1..10
        if page == 1:
            list_url = TOPIC_URL
        else:
            # Возможные форматы пагинации
            candidates = [
                f'{TOPIC_URL.rstrip("/")}/page/{page}/',
                f'{TOPIC_URL.rstrip("/")}/page/{page}',
                f'{TOPIC_URL}?page={page}',
            ]
            list_url = None
            for url in candidates:
                soup = fetch_soup(url, session)
                if soup:
                    # проверяем наличие статей на странице
                    links = collect_list_page_links(soup)
                    if links:
                        list_url = url
                        break
            if not list_url:
                print(f"Страница списка {page} не найдена или не содержит статей. Прерываю сбор.")
                break

        if list_url is None:
            list_url = TOPIC_URL  # fallback
        # Проверяем ещё раз, что страница действительно рабочая
        soup_check = fetch_soup(list_url, session)
        if not soup_check:
            print(f"Не удалось загрузить страницу списка: {list_url}")
            break

        # можно дополнительно проверить наличие статей здесь
        page_urls.append((page, list_url))

        time.sleep(random.uniform(0.5, 1.2))

    print(f"Найдено базовых страниц для обработки: {len(page_urls)}")

    # 2) Собираем ссылки на статьи со всех страниц
    article_links = []
    seen = set()
    for page_num, list_url in page_urls:
        soup = fetch_soup(list_url, session)
        if not soup:
            continue
        for href in collect_list_page_links(soup):
            if href not in seen:
                article_links.append((href, page_num))
                seen.add(href)
        time.sleep(random.uniform(0.5, 1.2))

    print(f"Найдено уникальных статей: {len(article_links)}")

    # 3) Получаем содержимое каждой статьи
    rows = []
    for article_url, page_num in article_links:
        art_soup = fetch_soup(article_url, session)
        if not art_soup:
            continue
        title, text = extract_title_and_text(art_soup)
        rows.append({'page_number': page_num, 'url': article_url, 'title': title, 'text': text})
        time.sleep(random.uniform(0.5, 1.2))

    # 4) Сохраняем в CSV
    df = pd.DataFrame(rows)
    df.to_csv('lifehacker_technology_articles.csv', index=False, encoding='utf-8')
    print(f"Сохранено {len(df)} статей в lifehacker_technology_articles.csv")

if __name__ == '__main__':
    main()

Найдено базовых страниц для обработки: 10
Найдено уникальных статей: 650


In [ ]:
from google.colab import drive
drive.mount('/content/drive')